# Comparison Summary Notebook

This notebook validates benchmark completeness and generates comparative outputs for all four implementations.


In [ ]:
from pathlib import Path
import os
import sys

if Path("/content/flax-kmeans").exists():
    os.chdir("/content/flax-kmeans")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the flax-kmeans repo root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)


In [ ]:
pip install -q -U pip
pip install -q -e ".[dev,jax,flash]"

In [ ]:
from pathlib import Path

from src.eval.notebook_harness import existing_run_ids, expected_run_ids, load_benchmark_matrix

MATRIX_PATH = Path("configs/experiments/benchmark_matrix.yaml")
matrix = load_benchmark_matrix(MATRIX_PATH)
RESULTS_ROOT = Path("results") / matrix.exp_name

IMPLEMENTATION_DEVICE_MAP = {
    "sklearn_kmeans": "cpu",
    "jax_kmeans": "tpu",
    "flashkmeans_wrapper": "gpu",
    "jax_flash_kmeans": "tpu",
}

expected = expected_run_ids(
    matrix_path=MATRIX_PATH,
    implementation_device_map=IMPLEMENTATION_DEVICE_MAP,
)
existing = existing_run_ids(RESULTS_ROOT)
missing = sorted(set(expected) - existing)

print("Expected run count:", len(expected))
print("Existing complete runs:", len(existing))
print("Missing run count:", len(missing))
if missing:
    raise RuntimeError(f"Missing benchmark artifacts for run_ids: {missing[:10]}")


In [ ]:
from src.eval.run_comparative_analysis import run_comparative_analysis

outputs = run_comparative_analysis(RESULTS_ROOT, RESULTS_ROOT)
for key, value in sorted(outputs.items()):
    print(f"{key}: {value}")


In [ ]:
report_path = RESULTS_ROOT / "plots" / "tradeoff_report.md"
print(report_path.read_text(encoding="utf-8"))
